In [ ]:
#model evaluation
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

In [ ]:
# Create and save the sample dataset
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# 40 Clinical Diagnostic Records with realistic borderline cases
data = {
    "patient_id": [f"P_{i:03d}" for i in range(1, 41)],
    "age": [
        45, 62, 54, 38, 71, 58, 49, 65, 41, 68,
        55, 60, 47, 73, 52, 66, 39, 63, 57, 50,
        44, 69, 53, 36, 75, 61, 48, 64, 42, 70,
        56, 59, 46, 72, 51, 67, 40, 62, 58, 49
    ],
    "systolic_bp": [
        120, 150, 135, 115, 165, 140, 128, 155, 118, 160,
        138, 145, 122, 170, 130, 158, 116, 152, 142, 125,
        121, 162, 136, 114, 175, 148, 126, 154, 119, 164,
        139, 144, 124, 168, 132, 156, 117, 150, 143, 127
    ],
    "cholesterol": [
        185, 260, 225, 175, 290, 240, 205, 275, 180, 285,
        230, 250, 195, 305, 215, 270, 170, 255, 245, 200,
        190, 280, 220, 165, 310, 258, 208, 265, 182, 295,
        235, 248, 198, 300, 218, 272, 172, 252, 242, 202
    ],
    "glucose": [
        85, 145, 110, 80, 175, 125, 95, 155, 82, 165,
        118, 135, 88, 190, 105, 160, 78, 140, 130, 92,
        86, 170, 112, 76, 200, 148, 98, 152, 84, 180,
        120, 138, 90, 185, 108, 162, 79, 142, 132, 94
    ],
    "bmi": [
        23.2, 31.5, 27.8, 21.4, 34.2, 29.1, 25.0, 32.8, 22.1, 33.5,
        28.4, 30.2, 24.3, 35.8, 26.5, 32.1, 21.8, 31.0, 29.5, 24.8,
        23.5, 33.8, 27.2, 20.8, 36.5, 31.2, 25.4, 32.4, 22.5, 34.0,
        28.8, 30.0, 24.6, 35.2, 26.8, 32.5, 21.5, 30.8, 29.2, 25.1
    ],
    "heart_rate": [
        68, 84, 75, 65, 92, 78, 72, 88, 66, 90,
        76, 82, 70, 95, 74, 86, 64, 83, 79, 71,
        69, 89, 77, 63, 98, 85, 73, 87, 67, 91,
        77, 81, 71, 94, 75, 88, 65, 82, 80, 72
    ],
    "has_condition": [
        0, 1, 1, 0, 1, 1, 1, 1, 0, 1,
        1, 0, 0, 1, 0, 1, 0, 1, 1, 0,
        0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
        1, 1, 0, 1, 0, 1, 0, 1, 1, 0
    ]
}

df = pd.DataFrame(data)

# Persist to local CSV
csv_path = "patient_diagnostic_data.csv"
df.to_csv(csv_path, index=False)
print(f"Dataset successfully created and saved to '{os.path.abspath(csv_path)}'")
print(f"Total Patients: {len(df)} | Total Features: {len(df.columns)}")

In [ ]:
# Load and inspect the dataset
df = pd.read_csv("patient_diagnostic_data.csv")

print(df.head())
print("\nDataset shape:", df.shape)
print("\nData types and missing values:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["has_condition"].value_counts())

In [ ]:
# Separate input features and target
feature_cols = [
    "age", "systolic_bp", "cholesterol",
    "glucose", "bmi", "heart_rate"
]

X = df[feature_cols]
y = df["has_condition"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# Train Logistic Regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression predictions:")
print(lr_pred)

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest predictions:")
print(rf_pred)

print("\nFeature importance:")
for feature, importance in zip(feature_cols, rf_model.feature_importances_):
    print(feature, ":", round(importance, 3))

In [ ]:
# Compare model evaluation metrics
def evaluate_model(y_true, y_pred, y_prob, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

lr_results = evaluate_model(y_test, lr_pred, lr_prob, "Logistic Regression")
rf_results = evaluate_model(y_test, rf_pred, rf_prob, "Random Forest")

results = pd.DataFrame([lr_results, rf_results])
display(results)

In [ ]:
# Confusion matrices and classification reports
lr_cm = confusion_matrix(y_test, lr_pred)
rf_cm = confusion_matrix(y_test, rf_pred)

print("Logistic Regression confusion matrix:")
print(lr_cm)
print("\nLogistic Regression classification report:")
print(classification_report(y_test, lr_pred, zero_division=0))

print("\nRandom Forest confusion matrix:")
print(rf_cm)
print("\nRandom Forest classification report:")
print(classification_report(y_test, rf_pred, zero_division=0))

In [ ]:
# Plot ROC curves to compare both models
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

plt.plot(lr_fpr, lr_tpr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, lr_prob):.2f})")
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_prob):.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Check how changing the threshold affects Logistic Regression
thresholds = [0.3, 0.5, 0.7]

for threshold in thresholds:
    threshold_pred = (lr_prob >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Accuracy:", round(accuracy_score(y_test, threshold_pred), 3))
    print("Precision:", round(precision_score(y_test, threshold_pred, zero_division=0), 3))
    print("Recall:", round(recall_score(y_test, threshold_pred, zero_division=0), 3))
    print("F1-score:", round(f1_score(y_test, threshold_pred, zero_division=0), 3))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, threshold_pred))

# Explain the concepts in your own words

1. **What is a confusion matrix?**  
   A confusion matrix is a table that compares actual values with predicted values. It shows correct and incorrect predictions.

2. **What is the difference between precision and recall?**  
   Precision tells us how many predicted positive cases are actually positive. Recall tells us how many actual positive cases the model correctly identifies.

3. **What is a false positive?**  
   A false positive happens when the model predicts a positive class, but the actual class is negative.

4. **What is a false negative?**  
   A false negative happens when the model predicts a negative class, but the actual class is positive.

5. **Why is accuracy not always enough?**  
   Accuracy can be misleading when one class has many more examples than another. Precision, recall, F1-score and the confusion matrix give more information.

6. **What does ROC-AUC measure?**  
   ROC-AUC measures how well a model separates positive and negative classes across different classification thresholds. A higher value generally indicates better separation.

7. **What happens when we lower the classification threshold?**  
   The model predicts the positive class more often. This can increase recall but may also increase false positives.

8. **Why do we evaluate using test data?**  
   Test data contains examples the model did not use during training. It helps us estimate how well the model performs on unseen data.

9. **Why do we scale data for Logistic Regression?**  
   Scaling puts numerical features on a similar range and can help Logistic Regression train more effectively.

10. **What is the difference between Logistic Regression and Random Forest?**  
    Logistic Regression uses a mathematical relationship to estimate class probabilities. Random Forest combines predictions from many decision trees.

**Note:** The patient dataset in this notebook is a sample educational dataset, not real medical data.